A layer published in one AWS account can be used by functions in other accounts, once you grant them access. This page shows how to grant and revoke that access for one account, a whole AWS organization, or everyone.

## Setup

This page runs against [moto](https://docs.getmoto.org/), which fakes AWS in memory, so every output below is real and needs no AWS account. In your own code, skip the moto cell and create the client from your own boto3 session.

In [1]:
# Demo only: fake AWS in memory. Skip this cell in your own code.
import moto

mock_aws = moto.mock_aws()
mock_aws.start()

In [2]:
import boto3

boto_ses = boto3.Session(region_name="us-east-1")
lambda_client = boto_ses.client("lambda")

Publish one layer version:

In [3]:
layer_name = "my_layer"

response = lambda_client.publish_layer_version(
    LayerName=layer_name,
    Content={"ZipFile": b"hello"},
)
version_number = response["Version"]

import json
import simple_aws_lambda.api as sal

## Who Can Be Granted Access

## Grant Access to an AWS Account

In [4]:
sal.grant_aws_account_or_aws_organization_lambda_layer_version_access(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=version_number,
    principal="111122223333",
)

The layer version policy now has one statement:

In [5]:
response = lambda_client.get_layer_version_policy(
    LayerName=layer_name,
    VersionNumber=version_number,
)
policy = json.loads(response["Policy"])
for statement in policy["Statement"]:
    print(statement["Sid"], statement["Principal"])

allow-111122223333-GetLayerVersion 111122223333


## Granting Twice Is Safe

The statement id is derived from the principal and the action, so granting the same principal again reuses the same id. AWS rejects the duplicate with `ResourceConflictException`, and the function ignores it: the access is already there. So you can call it on every deployment.

## Grant Access to an Organization and to Everyone

In [6]:
sal.grant_aws_account_or_aws_organization_lambda_layer_version_access(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=version_number,
    principal="o-1a2b3c",
)
sal.grant_aws_account_or_aws_organization_lambda_layer_version_access(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=version_number,
    principal="*",
)

An organization is granted as principal `*` limited by `OrganizationId`. A statement id may not contain `*`, so the public grant is named `public`:

In [7]:
response = lambda_client.get_layer_version_policy(
    LayerName=layer_name,
    VersionNumber=version_number,
)
policy = json.loads(response["Policy"])
for statement in policy["Statement"]:
    print(statement["Sid"], statement["Principal"], statement.get("OrganizationId"))

allow-111122223333-GetLayerVersion 111122223333 None
allow-o-1a2b3c-GetLayerVersion * o-1a2b3c
allow-public-GetLayerVersion * None


## Revoke Access

In [8]:
for principal in ["111122223333", "o-1a2b3c", "*"]:
    sal.revoke_aws_account_or_aws_organization_lambda_layer_version_access(
        lambda_client=lambda_client,
        layer_name=layer_name,
        version_number=version_number,
        principal=principal,
    )

Revoking access that does not exist does nothing, so it is just as safe to repeat. With every statement removed, AWS deletes the policy itself:

In [9]:
sal.revoke_aws_account_or_aws_organization_lambda_layer_version_access(
    lambda_client=lambda_client,
    layer_name=layer_name,
    version_number=version_number,
    principal="111122223333",
)

In [10]:
try:
    lambda_client.get_layer_version_policy(
        LayerName=layer_name,
        VersionNumber=version_number,
    )
except lambda_client.exceptions.ResourceNotFoundException as e:
    print(f"{type(e).__name__}: {e}")

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the GetLayerVersionPolicy operation: The resource you requested does not exist.


Stop moto when the demo is done:

In [11]:
mock_aws.stop()